# 实验：淘宝归因订单数据准备

## 目标

从提供的淘宝归因订单工作簿中，构建可复现、可用于后续分析的商品订单明细、订单汇总和 SKU 标签维表。

## 完成标准

- 7 份月度订单文件字段一致，且在不修改原始文件的前提下完成合并。
- 每条商品订单明细均可按商品 ID 匹配一条 SKU 标签记录。
- 订单汇总的归因付款金额与商品订单明细的归因付款金额能够对账。
- 数据质量检查覆盖缺失字段、非完整月份、重复模式、付款金额合理性和订单状态歧义。


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
assert (PROJECT_ROOT / 'src' / 'build_datasets.py').exists(), '请从项目根目录或 notebooks 目录运行此 Notebook。'

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
QUALITY_DIR = PROJECT_ROOT / 'outputs' / 'data_quality'
PROJECT_ROOT


## 执行步骤

1. 使用本地原始数据目录运行数据准备脚本。
2. 读取生成的 3 张分析表。
3. 对账归因付款金额，并验证必需的数据质量检查。
4. 在定义后续指标前，抽查多商品订单、混合状态订单和零付款商品明细。

数据来源是归因订单明细，不是完整广告、客户、结算或利润数据。


In [ ]:
command = [sys.executable, str(PROJECT_ROOT / 'src' / 'build_datasets.py')]
run = subprocess.run(command, cwd=PROJECT_ROOT, check=True, text=True, capture_output=True)
print(run.stdout)


## 读取生成的数据表

商品 ID、淘宝订单编号和推广位 ID 均以字符串读取，避免长数字 ID 的精度丢失。


In [ ]:
id_dtypes = {'商品ID': 'string', '淘宝订单编号': 'string', '推广位ID': 'string'}
order_lines = pd.read_csv(PROCESSED_DIR / 'order_lines_clean.csv', dtype=id_dtypes)
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv', dtype={'淘宝订单编号': 'string'})
sku_dimension = pd.read_csv(PROCESSED_DIR / 'sku_dimension.csv', dtype={'商品ID': 'string'})
quality_summary = json.loads((QUALITY_DIR / 'data_quality_summary.json').read_text(encoding='utf-8'))

overview = pd.Series({
    '商品订单明细数': len(order_lines),
    '订单汇总记录数': len(orders),
    'SKU 维表记录数': len(sku_dimension),
    '归因付款金额': round(order_lines['付款金额'].sum(), 2),
    'SKU 标签匹配率（%）': quality_summary['sku_dimension']['order_line_tag_match_rate_percent'],
})
overview


## 自动校验

这些检查用于确认生成的数据表之间可以对账；它们不验证 `付款金额` 的商业含义，也不证明订单状态的完整生命周期。


In [ ]:
assert len(order_lines) == quality_summary['line_table']['row_count']
assert len(orders) == quality_summary['order_table']['row_count']
assert order_lines['一级标签'].notna().all(), '存在未匹配的 SKU 标签，需复核。'
assert round(order_lines['付款金额'].sum(), 2) == round(orders['attributed_payment_amount'].sum(), 2)
assert not order_lines.duplicated().any(), '存在完全重复的商品订单明细，需复核。'
assert (order_lines['商品数量'] >= 0).all()
assert (order_lines['付款金额'] >= 0).all()
assert (order_lines['付款金额'] <= order_lines['list_amount']).all()
assert set(orders['order_status_classification']).issubset({
    'invalid_order', 'non_invalid_order', 'mixed_status_order'
})

pd.DataFrame({'检查项': quality_summary['checks'].keys(), '通过': quality_summary['checks'].values()})


## 人工抽查样本

进入业务分析前，请回到原始 Excel 抽查多商品订单、混合状态订单和零付款商品明细。抽查目的不是寻找预设答案，而是确认订单级汇总规则是否符合源数据的业务含义。


In [ ]:
sample_order_ids = orders.loc[
    orders['order_line_count'].gt(1) | orders['order_status_classification'].eq('mixed_status_order'),
    '淘宝订单编号',
].head(5)

manual_review_sample = (
    order_lines.loc[order_lines['淘宝订单编号'].isin(sample_order_ids)]
    .sort_values(['淘宝订单编号', '商品ID'])
    [['淘宝订单编号', '商品ID', '商品标题', '商品数量', '付款金额', '订单状态', '推广位名称']]
)
manual_review_sample


In [ ]:
result = {
    '点击日期范围': [quality_summary['line_table']['date_min'], quality_summary['line_table']['date_max']],
    '完整月份': [
        row['click_month'] for row in quality_summary['source_month_coverage'] if row['is_complete_month']
    ],
    '非完整月份': [
        row['click_month'] for row in quality_summary['source_month_coverage'] if not row['is_complete_month']
    ],
    '混合状态订单数': quality_summary['order_table']['mixed_status_orders'],
    '零付款商品明细数': quality_summary['line_table']['zero_payment_rows'],
    '不纳入财务分析的字段': ['结算时间', '结算金额', '佣金金额', '淘宝子订单号'],
}
result


## 下一步

- 确认失效订单、非失效订单和混合状态订单的最终业务口径。
- 在计算商品或推广位表现前，先写清指标定义。
- 普通月度环比仅使用完整月份。
- 不得将归因付款金额表述为最终营收、结算收入、利润、佣金或 ROI。
